<a href="https://colab.research.google.com/github/nalinkai/Data-Science-Project-Lifecycle/blob/main/Feature_Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

# Load the data
df = pd.read_csv('/content/drive/MyDrive/Data Science Life Cycle Coursework/Hotel-A-train_processed_data.csv')
print(df.head())

# Display basic info
print(df.info())
print(df.head())

   Reservation-id Gender  Age         Ethnicity Educational_Level  \
0        39428300      f   40            latino              grad   
1        77491756      f   49            latino        mid-school   
2        73747291      f   42         caucasian              grad   
3        67301739      m   25  african american           college   
4        77222321      f   62            latino       high-school   

        Income Country_region      Hotel_Type Expected_checkin  \
0         <25k          north      city hotel       2015-07-01   
1  50k -- 100k           east      city hotel       2015-07-01   
2         <25k           east      city hotel       2015-07-02   
3        >100k          south  airport hotels       2015-07-02   
4    25k --50k           east          resort       2015-07-03   

  Expected_checkout  ... Previous_Cancellations  Deposit_type  \
0        2015-07-02  ...                     no    no deposit   
1        2015-07-02  ...                     no    refunda

Feature Engineering - Create New Features

In [ ]:
# Convert date columns to datetime
date_columns = ['Expected_checkin', 'Expected_checkout', 'Booking_date']

for col in date_columns:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

# Extract date-based features
# Check-in date features
df['checkin_month'] = df['Expected_checkin'].dt.month
df['checkin_dayofweek'] = df['Expected_checkin'].dt.dayofweek  # Monday=0, Sunday=6

# Check-out date features
df['checkout_month'] = df['Expected_checkout'].dt.month
df['checkout_dayofweek'] = df['Expected_checkout'].dt.dayofweek

# Booking date features
df['booking_month'] = df['Booking_date'].dt.month
df['booking_day'] = df['Booking_date'].dt.day

# Seasonal features
df['is_summer'] = df['checkin_month'].isin([6, 7, 8]).astype(int)
df['is_holiday_season'] = df['checkin_month'].isin([12, 1, 7, 8]).astype(int)

# Drop original date columns (already processed)
df = df.drop(columns=date_columns, errors='ignore')

In [ ]:
df.head()

,Reservation-id,Gender,Age,Ethnicity,Educational_Level,Income,Country_region,Hotel_Type,Adults,Children,...,lead_time_days,stay_duration_days,checkin_month,checkin_dayofweek,checkout_month,checkout_dayofweek,booking_month,booking_day,is_summer,is_holiday_season
0,39428300,f,40,latino,grad,<25k,north,city hotel,2,2,...,41,1,7,2,7,3,5,21,1,1
1,77491756,f,49,latino,mid-school,50k -- 100k,east,city hotel,3,3,...,36,1,7,2,7,3,5,26,1,1
2,73747291,f,42,caucasian,grad,<25k,east,city hotel,3,3,...,3,4,7,3,7,0,6,29,1,1
3,67301739,m,25,african american,college,>100k,south,airport hotels,4,3,...,12,1,7,3,7,4,6,20,1,1
4,77222321,f,62,latino,high-school,25k --50k,east,resort,1,1,...,13,1,7,4,7,5,6,20,1,1


In [ ]:
df = df.drop(columns='is_summer', errors='ignore')

In [ ]:
df.head()

,Reservation-id,Gender,Age,Ethnicity,Educational_Level,Income,Country_region,Hotel_Type,Adults,Children,...,Room_Rate,lead_time_days,stay_duration_days,checkin_month,checkin_dayofweek,checkout_month,checkout_dayofweek,booking_month,booking_day,is_holiday_season
0,39428300,f,40,latino,grad,<25k,north,city hotel,2,2,...,218,41,1,7,2,7,3,5,21,1
1,77491756,f,49,latino,mid-school,50k -- 100k,east,city hotel,3,3,...,185,36,1,7,2,7,3,5,26,1
2,73747291,f,42,caucasian,grad,<25k,east,city hotel,3,3,...,119,3,4,7,3,7,0,6,29,1
3,67301739,m,25,african american,college,>100k,south,airport hotels,4,3,...,144,12,1,7,3,7,4,6,20,1
4,77222321,f,62,latino,high-school,25k --50k,east,resort,1,1,...,242,13,1,7,4,7,5,6,20,1


In [ ]:
# Convert 'Visted_Previously' and 'Previous_Cancellations' to numeric (0 or 1)
# Assuming 'no' means 0 and 'yes' means 1 for these columns.
# Check if column exists and is object type before mapping to avoid errors on already converted columns or non-existent columns.
if 'Visted_Previously' in df.columns and df['Visted_Previously'].dtype == 'object':
    df['Visted_Previously'] = df['Visted_Previously'].map({'yes': 1, 'no': 0}).fillna(0).astype(int)
    print(f"Converted 'Visted_Previously' to numeric: {df['Visted_Previously'].dtype}")
elif 'Visted_Previously' in df.columns: # If not object, it's already numeric, ensure it's int
    df['Visted_Previously'] = df['Visted_Previously'].astype(int)
    print(f"Ensured 'Visted_Previously' is int: {df['Visted_Previously'].dtype}")


if 'Previous_Cancellations' in df.columns and df['Previous_Cancellations'].dtype == 'object':
    df['Previous_Cancellations'] = df['Previous_Cancellations'].map({'yes': 1, 'no': 0}).fillna(0).astype(int)
    print(f"Converted 'Previous_Cancellations' to numeric: {df['Previous_Cancellations'].dtype}")
elif 'Previous_Cancellations' in df.columns: # If not object, it's already numeric, ensure it's int
    df['Previous_Cancellations'] = df['Previous_Cancellations'].astype(int)
    print(f"Ensured 'Previous_Cancellations' is int: {df['Previous_Cancellations'].dtype}")

# 1. Guest composition features
df['total_guests'] = df['Adults'] + df['Children'] + df['Babies']
df['is_family'] = ((df['Children'] + df['Babies']) > 0).astype(int)

# 2. Stay duration (already have, but ensure it's correct)
# stay_duration_days exists, but let's verify
if 'stay_duration_days' not in df.columns:
    df['stay_duration_days'] = (df['Expected_checkout'] - df['Expected_checkin']).dt.days

# 3. Lead time features
# lead_time_days exists, but we can create additional features
df['lead_time_category'] = pd.cut(df['lead_time_days'],
                                   bins=[0, 7, 30, 90, 365],
                                   labels=['Very Short', 'Short', 'Medium', 'Long'])
# Convert lead_time_category to numerical codes
df['lead_time_category'] = df['lead_time_category'].cat.codes

# 4. Price-related features
df['total_cost'] = df['Room_Rate'] * df['stay_duration_days']
df['discounted_rate'] = df['Room_Rate'] * (1 - df['Discount_Rate']/100)

# 5. Previous behavior features
df['previous_stays_cancelled_ratio'] = df['Previous_Cancellations'] / (df['Visted_Previously'] + 1)

# 8. Booking channel features
if 'Booking_channel' in df.columns and df['Booking_channel'].dtype == 'object':
    # Create flags for important channels
    df['is_online_booking'] = (df['Booking_channel'] == 'online').astype(int)
    df['is_direct_booking'] = (df['Booking_channel'] == 'direct').astype(int)
    df['is_agent_booking'] = (df['Booking_channel'] == 'agent').astype(int)

# 10. Promotion features
if 'Use_Promotion' in df.columns:
    df['used_promotion'] = df['Use_Promotion'].map({'yes': 1, 'no': 0})

# 11. Interaction features
df['price_per_person'] = df['Room_Rate'] / (df['total_guests'] + 1)
df['lead_time_price_interaction'] = df['lead_time_days'] * df['Room_Rate']
df['cancellation_risk'] = df['Previous_Cancellations'] * df['Discount_Rate']

# 12. Weekend vs weekday stay
df['stays_include_weekend'] = ((df['checkin_dayofweek'] >= 5) |
                                (df['checkout_dayofweek'] <= 1)).astype(int)

Ensured 'Visted_Previously' is int: int64
Ensured 'Previous_Cancellations' is int: int64


In [ ]:
df.head()

,Reservation-id,Gender,Age,Ethnicity,Educational_Level,Income,Country_region,Hotel_Type,Adults,Children,...,discounted_rate,previous_stays_cancelled_ratio,is_online_booking,is_direct_booking,is_agent_booking,used_promotion,price_per_person,lead_time_price_interaction,cancellation_risk,stays_include_weekend
0,39428300,f,40,latino,grad,<25k,north,city hotel,2,2,...,196.2,0.0,1,0,0,1,43.600000,8938,0,0
1,77491756,f,49,latino,mid-school,50k -- 100k,east,city hotel,3,3,...,185.0,0.0,1,0,0,0,26.428571,6660,0,0
2,73747291,f,42,caucasian,grad,<25k,east,city hotel,3,3,...,119.0,0.0,1,0,0,0,17.000000,357,0,1
3,67301739,m,25,african american,college,>100k,south,airport hotels,4,3,...,136.8,0.0,0,0,1,1,18.000000,1728,0,0
4,77222321,f,62,latino,high-school,25k --50k,east,resort,1,1,...,217.8,0.0,0,1,0,1,80.666667,3146,0,0


Encode Categorical Variables

In [ ]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer

# Identify categorical columns (that are still objects) that need to be encoded
current_categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

# Remove target if it's still in categorical
if 'target' in current_categorical_cols:
    current_categorical_cols.remove('target')
if 'Reservation_Status' in current_categorical_cols:
    current_categorical_cols.remove('Reservation_Status')

print(f"\nCategorical columns to one-hot encode: {current_categorical_cols}")

# Apply one-hot encoding to all identified categorical columns
df = pd.get_dummies(df, columns=current_categorical_cols, drop_first=True)


Categorical columns to one-hot encode: ['Educational_Level', 'Meal_Type', 'Deposit_type', 'Booking_channel', 'Required_Car_Parking', 'Use_Promotion']


In [ ]:
df.head()

,Reservation-id,Age,Educational_Level,Income,Adults,Children,Babies,Meal_Type,Visted_Previously,Previous_Cancellations,...,stays_include_weekend,Hotel_Type_city hotel,Hotel_Type_resort,Ethnicity_asian american,Ethnicity_caucasian,Ethnicity_latino,Country_region_north,Country_region_south,Country_region_west,Gender_m
0,39428300,40.0,grad,<25k,2.0,2.0,0.0,bb,0.0,-1.387779e-17,...,0.0,True,False,False,False,True,True,False,False,False
1,77491756,49.0,mid-school,50k -- 100k,3.0,3.0,0.0,bb,0.0,-1.387779e-17,...,0.0,True,False,False,False,True,False,False,False,False
2,73747291,42.0,grad,<25k,3.0,3.0,0.0,bb,0.0,-1.387779e-17,...,1.0,True,False,False,True,False,False,False,False,False
3,67301739,25.0,college,>100k,4.0,3.0,0.0,bb,0.0,-1.387779e-17,...,0.0,False,False,False,False,False,False,True,False,True
4,77222321,62.0,high-school,25k --50k,1.0,1.0,0.0,bb,0.0,-1.387779e-17,...,0.0,False,True,False,False,True,False,False,False,False


Handle Numeric Features - Scaling

In [ ]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# Select columns
cols = ['Income', 'Discount_Rate', 'Room_Rate']

# Map 'Income' categorical values to numerical values
income_mapping = {
    '<25k': 0,
    '25k --50k': 1,
    '50k -- 100k': 2,
    '>100k': 3
}
df['Income'] = df['Income'].map(income_mapping).fillna(0).astype(int)

# Apply scaling
scaler = MinMaxScaler()
df[cols] = scaler.fit_transform(df[cols])

df.head()

,Reservation-id,Age,Educational_Level,Income,Adults,Children,Babies,Meal_Type,Visted_Previously,Previous_Cancellations,...,stays_include_weekend,Hotel_Type_city hotel,Hotel_Type_resort,Ethnicity_asian american,Ethnicity_caucasian,Ethnicity_latino,Country_region_north,Country_region_south,Country_region_west,Gender_m
0,39428300,40.0,grad,0.000000,2.0,2.0,0.0,bb,0.0,-1.387779e-17,...,0.0,True,False,False,False,True,True,False,False,False
1,77491756,49.0,mid-school,0.666667,3.0,3.0,0.0,bb,0.0,-1.387779e-17,...,0.0,True,False,False,False,True,False,False,False,False
2,73747291,42.0,grad,0.000000,3.0,3.0,0.0,bb,0.0,-1.387779e-17,...,1.0,True,False,False,True,False,False,False,False,False
3,67301739,25.0,college,1.000000,4.0,3.0,0.0,bb,0.0,-1.387779e-17,...,0.0,False,False,False,False,False,False,True,False,True
4,77222321,62.0,high-school,0.333333,1.0,1.0,0.0,bb,0.0,-1.387779e-17,...,0.0,False,True,False,False,True,False,False,False,False


Feature Selection

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

# Encode the target variable 'Reservation_Status'
le = LabelEncoder()
df['Reservation_Status_encoded'] = le.fit_transform(df['Reservation_Status'])

# Prepare feature matrix and target
# Drop 'Reservation-id' (identifier), 'Reservation_Status' (original string target),
# and the new encoded target column from features X
X = df.drop(columns=['Reservation-id', 'Reservation_Status', 'Reservation_Status_encoded'], errors='ignore')
y = df['Reservation_Status_encoded'] # Use the encoded target

# Method 1: Correlation with target
# This method expects a numerical target, which 'Reservation_Status_encoded' now is.
feature_corr = pd.DataFrame({
    'feature': X.columns,
    'correlation': [X[col].corr(y) for col in X.columns]
})
feature_corr = feature_corr.sort_values('correlation', key=abs, ascending=False)
print("\nTop 20 features by correlation:")
print(feature_corr.head(20))

# Method 2: Mutual Information
# mutual_info_classif is suitable for categorical targets (which encoded target represents).
mi_scores = mutual_info_classif(X, y, random_state=42)
mi_df = pd.DataFrame({'feature': X.columns, 'mi_score': mi_scores})
mi_df = mi_df.sort_values('mi_score', ascending=False)
print("\nTop 20 features by Mutual Information:")
print(mi_df.head(20))

# Method 3: Random Forest Feature Importance
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X, y)
importance_df = pd.DataFrame({
    'feature': X.columns,
    'importance': rf.feature_importances_
})
importance_df = importance_df.sort_values('importance', ascending=False)
print("\nTop 20 features by Random Forest Importance:")
print(importance_df.head(20))

# Select top features (e.g., top 50 based on importance)
top_features = importance_df.head(50)['feature'].tolist()
X_selected = X[top_features]

print(f"\nSelected {len(top_features)} features for modeling")

/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]



Top 20 features by correlation:
                           feature  correlation
44                    Meal_Type_fb    -0.051664
16                     booking_day     0.018474
12               checkin_dayofweek    -0.016314
20              lead_time_category    -0.015758
4                           Babies     0.015385
31           stays_include_weekend    -0.014694
13                  checkout_month     0.013801
11                   checkin_month     0.013709
18                    total_guests     0.013472
35             Ethnicity_caucasian    -0.013312
33               Hotel_Type_resort    -0.012228
2                           Adults     0.011559
43    Educational_Level_mid-school     0.010036
40                        Gender_m     0.009039
47         Deposit_type_refundable    -0.008921
15                   booking_month     0.008878
29     lead_time_price_interaction     0.007943
34        Ethnicity_asian american    -0.007199
6           Previous_Cancellations     0.006429
23  pre

In [ ]:
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek
from collections import Counter

print("\nClass distribution before balancing:")
print(Counter(y))

# Option 1: SMOTE (Synthetic Minority Over-sampling)
smote = SMOTE(random_state=42)
X_balanced, y_balanced = smote.fit_resample(X_selected, y)

# Option 2: SMOTE + Tomek links (better for borderline cases)
# smote_tomek = SMOTETomek(random_state=42)
# X_balanced, y_balanced = smote_tomek.fit_resample(X_selected, y)

print("\nClass distribution after balancing:")
print(Counter(y_balanced))


Class distribution before balancing:
Counter({1: 20774, 0: 4108, 2: 2107})

Class distribution after balancing:
Counter({1: 20774, 0: 20774, 2: 20774})


In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold

# Split with stratification to maintain class distribution
X_train, X_test, y_train, y_test = train_test_split(
    X_balanced, y_balanced,
    test_size=0.2,
    random_state=42,
    stratify=y_balanced
)

print(f"\nTraining set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")
print(f"\nTraining class distribution:")
print(Counter(y_train))
print(f"\nTest class distribution:")
print(Counter(y_test))


Training set size: (49857, 50)
Test set size: (12465, 50)

Training class distribution:
Counter({1: 16619, 2: 16619, 0: 16619})

Test class distribution:
Counter({1: 4155, 0: 4155, 2: 4155})


In [ ]:
df.shape

(26989, 55)

In [ ]:
import pandas as pd
from google.colab import files

# Save file
df.to_csv("/content/drive/MyDrive/Data Science Life Cycle Coursework/Hotel-A-train_finalized.csv", index=False)

In [ ]:
# Drop the column
df = df.drop('total_guests', axis=1)

NameError: name 'df' is not defined